In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import config
from instruments.attenuator import Attenuator
from zhinst.qcodes import SHFQC
import readout
from zhinst.toolkit import CommandTable

In [ ]:
class Rabi:
    def __init__(self, qubit_freq, readout_p, detunings, raw_data_dir, plots_dir):
        self.qubit_freq = qubit_freq
        self.detunings = detunings
        self.raw_data_dir = raw_data_dir
        self.plots_dir = plots_dir
        self.shfqc = SHFQC("DEVXXXX", "localhost")
        self.atten = Attenuator(config.LABBER_SERVER, config.LABBER_SERVER_TIMEOUT, config.ATTENUATOR_NAME, config.ATTENUATOR_INTERFACE, config.ATTENUATOR_ADDRESS)
        self.atten_wait = config.ATTENUATOR_WAIT_S
        self.atten.set_attenuation(readout_p, wait=self.atten_wait)
        self.CHANNEL = 0
    

    def configure(self):
        self.shfqc.sgchannels[0].configure_channel(
        enable=True,
        output_range=0,
        center_frequency=self.qubit_freq,
        rf_path=True
        )
        with self.shfqc.set_transaction():
            self.shfqc.sgchannels[self.CHANNEL].oscs[0].freq(self.detunings[0])
            self.shfqc.sgchannels[self.CHANNEL].sines[0].oscselect(0)
            self.shfqc.sgchannels[self.CHANNEL].sines[0].harmonic(1)
            self.shfqc.sgchannels[self.CHANNEL].sines[0].phaseshift(0)
            self.shfqc.sgchannels[self.CHANNEL].awg.modulation.enable(1)
            # Set marker source
            self.shfqc.sgchannels[self.CHANNEL].marker.source(0)

    def load_seqc(self):
        SEQUENCER_CODE = """\
        // Define constants in time domain
        const t_wid = 50e-9;
        const t_len = t_wid*8;
        const amp = 1;
        const n_amps = 101;
        const n_aves = 1000;
        const t_readout = 1e-6;

        // Convert to samples
        const s_rate = 2.0e9;
        const s_wid = t_wid*s_rate;
        const s_len= round(s_rate*t_len/16)*16; //Account for waveform granularity of 16 samples
        const s_readout = round(s_rate*t_readout/16)*16;

        // Define waveform and assign index
        wave w = gauss(s_len, amp, s_len/2, s_wid);
        assignWaveIndex(1,2,w,1,2,w,0);

        // Reset oscillator phases and trigger scope
        resetOscPhase();
        setTrigger(1);
        setTrigger(0);

        //First Rabi amplitude and readout
        executeTableEntry(0);
        playZero(s_readout);

        //Increment Rabi amplitude each loop iteration
        repeat (n_amps-1) {
          resetOscPhase();
          setTrigger(1);
          setTrigger(0);

          executeTableEntry(1);
          //Readout window
          playZero(s_readout);
        }
        """
        self.shfqc.sgchannels[self.CHANNEL].awg.load_sequencer_program(SEQUENCER_CODE)

    def load_ct(self):
        ct_schema = self.shfqc.sgchannels[self.CHANNEL].awg.commandtable.load_validation_schema()
        ct = CommandTable(ct_schema)

        ct.table[0].waveform.index = 0
        ct.table[0].amplitude00.value = 0.0
        ct.table[0].amplitude00.increment = False
        ct.table[0].amplitude01.value = -0.0
        ct.table[0].amplitude01.increment = False
        ct.table[0].amplitude10.value = 0.0
        ct.table[0].amplitude10.increment = False
        ct.table[0].amplitude11.value = 0.0
        ct.table[0].amplitude11.increment = False

        ct.table[1].waveform.index = 0
        ct.table[1].amplitude00.value = 0.007
        ct.table[1].amplitude00.increment = True
        ct.table[1].amplitude01.value = -0.007
        ct.table[1].amplitude01.increment = True
        ct.table[1].amplitude10.value = 0.007
        ct.table[1].amplitude10.increment = True
        ct.table[1].amplitude11.value = 0.007
        ct.table[1].amplitude11.increment = True

        self.shfqc.sgchannels[self.CHANNEL].awg.commandtable.upload_to_device(ct)
    
    def chevron(self):
        data = np.zeros((len(self.detunings), 101, 2))
        for i, osc_freq in enumerate(self.detunings):
            self.shfqc.sgchannels[self.CHANNEL].oscs[0].freq(osc_freq)
            readout.arm()
            self.shfqc.sgchannels[self.CHANNEL].awg.enable_sequencer(single=True)
            self.shfqc.sgchannels[self.CHANNEL].awg.wait_done()
            readout_data = readout.readout()
            data[i] = readout_data

        return data

